# M05B · UniProt, PDB & KEGG: Structure, Function & Pathways
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M05_Databases_and_Scraping/M05B_UniProt_PDB_KEGG_Solutions.ipynb)

⏱ ~120 min · Difficulty: ★★☆ · Prerequisites: M01A (Core Python), M02C (Biopython), M05A (NCBI Entrez)


## 1. Learning Outcomes

By the end of this notebook, you will be able to:

- **Query**: Query the UniProtKB REST API programmatically using `requests`, retrieve structured tabular metadata, and parse FASTA and Swiss-Prot formats.
- **Traverse**: Navigate the Structural SMCRA (Structure-Model-Chain-Residue-Atom) hierarchy in Biopython's `Bio.PDB` parser to inspect 3D atomic coordinates.
- **Contrast**: Compare legacy PDB column-fixed text format against the modern mmCIF dictionary-based standard and parse both using `PDBParser` and `MMCIFParser`.
- **Reconstruct**: Extract polypeptide sequences from 3D structures using `Bio.PDB.PPBuilder` and compute Euclidean distance contact networks.
- **Integrate**: Query the KEGG REST API (`Bio.KEGG.REST`) and assemble an end-to-end multi-database cross-referencing pipeline linking Gene, Protein, Structure, and Pathway.

## 2. Prerequisites


Before starting this notebook, you should be familiar with:
- Biopython `SeqIO` sequence parsing ([`M02C`](file:///modules/M02_File_Handling/M02C_Biopython_SeqIO_GenBank.ipynb)).
- NCBI Entrez and accession formats ([`M05A`](file:///modules/M05_Databases_and_Scraping/M05A_Entrez_GenBank_PubMed.ipynb)).
- Basic Euclidean geometry in 3D space: $d = \sqrt{(x_2-x_1)^2 + (y_2-y_1)^2 + (z_2-z_1)^2}$.


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install biopython seaborn requests beautifulsoup4 lxml python-dotenv
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root robustly from modules/ or repo root
    p = Path.cwd()
    while p != p.parent and not (p / "COURSE_PLAN.md").exists():
        p = p.parent
    ROOT = p if (p / "COURSE_PLAN.md").exists() else Path.cwd().resolve()

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M05B"
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests, io
from Bio.PDB import PDBParser, MMCIFParser, PPBuilder
from Bio.KEGG import REST as KEGG_REST
from Bio import SeqIO

print(f"Python: {sys.version.split()[0]}")
print(f"Working Directory: {Path.cwd()}")
print(f"DATA_DIR: {DATA_DIR} (exists: {DATA_DIR.exists()})")
print(f"OUT_DIR:  {OUT_DIR} (ready)")

## 5B.1 UniProt REST API & Swiss-Prot Flat Files

### 4.1.1 Theory & Biological Context

The **Universal Protein Resource (UniProt)** is the world's most comprehensive catalog of protein sequence and functional annotation.

### UniProtKB Architecture:
1. **UniProtKB/Swiss-Prot**: Manually reviewed, high-quality, non-redundant computational and literature-curated records (e.g. `P04637` for human p53).
2. **UniProtKB/TrEMBL**: Automatically translated and computationally annotated EMBL/GenBank coding sequences awaiting manual curation.

### The Modern UniProt REST API:
UniProt provides a high-throughput REST API endpoint:
`https://rest.uniprot.org/uniprotkb/search`

Key URL Query Parameters:
- `query`: Boolean search syntax, e.g. `reviewed:true AND gene:TP53 AND organism_id:9606`
- `format`: Response format (`tsv`, `json`, `fasta`, `xml`)
- `fields`: Comma-separated list of attributes (`accession,id,gene_names,length,cc_function,sequence`)
- `size`: Maximum records to return per page.

### 4.1.2 Worked Example

In [ ]:
# 5B.1 Worked Example: Querying UniProt REST API and Parsing TSV
uniprot_url = "https://rest.uniprot.org/uniprotkb/search"
query_params = {
    "query": "reviewed:true AND gene:TP53 AND organism_id:9606",
    "format": "tsv",
    "fields": "accession,id,gene_names,length,cc_function",
    "size": 5
}

try:
    print("Querying UniProtKB REST API...")
    resp = requests.get(uniprot_url, params=query_params, timeout=10)
    resp.raise_for_status()
    
    # Read TSV string directly into pandas
    df_uniprot = pd.read_csv(io.StringIO(resp.text), sep="\t")
    print(f"Retrieved {len(df_uniprot)} record(s):")
    print(df_uniprot[["Entry", "Entry Name", "Gene Names", "Length"]].to_string())
except Exception as e:
    print(f"Network query skipped ({e}). Using offline mock data.")
    mock_tsv = """Entry\tEntry Name\tGene Names\tLength\tFunction [CC]
P04637\tP53_HUMAN\tTP53 P53\t393\tActs as a tumor suppressor in all types of cancer.
"""
    df_uniprot = pd.read_csv(io.StringIO(mock_tsv), sep="\t")
    print(df_uniprot.to_string())

### 4.1.3 ✏️ Try It: UniProt Summary Fetcher

Write a function `fetch_uniprot_summary(gene: str, taxid: int = 9606) -> dict`:
- Query UniProt REST API for `reviewed:true AND gene:{gene} AND organism_id:{taxid}`.
- If network succeeds, extract `Entry`, `Entry Name`, `Length` (as int) from the first row.
- If network fails, return a simulated fallback dictionary: `{"accession": "P04637", "name": f"{gene}_HUMAN", "length": 393}`.
- Return the dictionary with keys `'accession'`, `'name'`, `'length'`.

In [ ]:
def fetch_uniprot_summary(gene: str, taxid: int = 9606) -> dict:
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"reviewed:true AND gene:{gene} AND organism_id:{taxid}",
        "format": "tsv",
        "fields": "accession,id,length",
        "size": 1
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text), sep="\t")
        if not df.empty:
            return {
                "accession": df.iloc[0]["Entry"],
                "name": df.iloc[0]["Entry Name"],
                "length": int(df.iloc[0]["Length"])
            }
    except Exception:
        pass
    return {"accession": "P04637", "name": f"{gene}_HUMAN", "length": 393}

#### 💡 Explanation & Design Notes

Querying the UniProt REST API with `format=tsv` and specific `fields` reduces payload size from megabytes of full XML down to a lightweight tabular record that parses directly into a DataFrame in milliseconds.

In [ ]:
# ✅ Test — run me. No output means you passed.
p53_info = fetch_uniprot_summary("TP53", 9606)
assert p53_info["accession"] == "P04637"
assert p53_info["length"] == 393
assert "P53" in p53_info["name"]
print("Section 5B.1 passed ✔")

### 4.1.4 ⚠️ Common Mistakes

- **Omitting reviewed:true in UniProt queries**: Without `reviewed:true`, searches return hundreds of unreviewed TrEMBL translations, redundant splice isoforms, and synthetic peptide constructs.
- **Hardcoding taxid instead of organism_id**: In the modern UniProt REST API, use `organism_id:9606` (NCBI Taxonomy ID for human) rather than free-text `human` to prevent matching viral proteins that mimic human genes.

## 5B.2 Macromolecular 3D Structure & The SMCRA Hierarchy

### 4.2.1 Theory & Biological Context

Biopython's `Bio.PDB` module parses 3D experimental macromolecular coordinates (X-ray crystallography, Cryo-EM, NMR).

### The Structural SMCRA Hierarchy:
Biopython represents 3D structures using an Object-Oriented nested containment hierarchy called **SMCRA**:
```
Structure  (Entire macromolecular complex, e.g. '1TUP')
  └── Model  (0-indexed model state; 1 model for X-ray, multiple for NMR)
        └── Chain  (Individual polypeptide or polynucleotide chain, e.g. 'A', 'B', 'D')
              └── Residue  (Amino acid, nucleotide, or heteroatom, e.g. ARG 248, ZN 300)
                    └── Atom  (Individual atom with 3D coordinate vector, e.g. CA, CB, N)
```

### Navigating SMCRA:
- `structure[0]`: Model 0.
- `model['A']`: Chain A.
- `chain[248]`: Residue with sequence identifier 248.
- `residue['CA']`: Alpha Carbon atom.
- `atom.get_coord()`: NumPy array `[x, y, z]` in Ångström ($\text{\AA}$) units.
- `atom1 - atom2`: Computes Euclidean distance in $\text{\AA}$ directly!

### 4.2.2 Worked Example

In [ ]:
# 5B.2 Worked Example: Inspecting 1TUP (p53-DNA complex) with Bio.PDB
pdb_file = DATA_DIR / "structures" / "1tup.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("1TUP", pdb_file)
model = structure[0]

print(f"Structure ID: {structure.id}")
print(f"Number of models: {len(structure)}")
print(f"Chains present: {[c.id for c in model]}")

# Inspect chain compositions
for chain in model:
    residues = list(chain.get_residues())
    # Separate standard amino acids/nucleotides from heteroatoms (HETATM)
    standard_res = [r for r in residues if r.id[0] == " "]
    het_res = [r for r in residues if r.id[0] != " "]
    print(f"  Chain {chain.id}: {len(standard_res)} standard residues, {len(het_res)} heteroatoms/ligands")

# Extract polypeptide sequence using PPBuilder
ppb = PPBuilder()
peptides = ppb.build_peptides(model["A"])
if peptides:
    print(f"Chain A Polypeptide sequence ({len(peptides[0])} aa):")
    print(f"  {peptides[0].get_sequence()[:50]}...")

### 4.2.3 ✏️ Try It: Zinc Atom Counter

Write a function `count_zinc_atoms(structure) -> int`:
- Iterates over all atoms in the structure.
- Counts and returns the total number of Zinc atoms (where `atom.element == "ZN"` or residue name is `"ZN"`).

In [ ]:
def count_zinc_atoms(structure) -> int:
    count = 0
    for atom in structure.get_atoms():
        if atom.element.upper() == "ZN" or atom.get_parent().get_resname().strip() == "ZN":
            count += 1
    return count

#### 💡 Explanation & Design Notes

The tumor suppressor p53 core domain coordinates a single structural Zinc ion ($Zn^{2+}$) via Cys176, His179, Cys238, and Cys242. In 1TUP, the asymmetric unit contains 3 p53 monomers (Chains A, B, C), each containing 1 catalytic zinc atom.

In [ ]:
# ✅ Test — run me. No output means you passed.
zn_count = count_zinc_atoms(structure)
assert zn_count == 3, f"Expected 3 Zinc atoms in 1TUP (one per p53 core domain monomer), got {zn_count}"
print("Section 5B.2 passed ✔")

### 4.2.4 ⚠️ Common Mistakes

- **Treating residue IDs as integers**: In Biopython, `residue.id` is a 3-tuple `(hetflag, resseq, icode)`. Accessing sequence number is `residue.id[1]`, not `residue.id`.
- **Assuming chain.get_residues() contains only amino acids**: `get_residues()` returns waters (`HOH`), ligands (`ZN`), and nucleic acids unless filtered by `r.id[0] == ' '`.

## 5B.3 PDB vs mmCIF Format Deep-Dive

### 4.3.1 Theory & Biological Context

Macromolecular crystallography originally recorded coordinates in the fixed-column text **PDB format** (1970s).

### Limitations of Legacy PDB Format:
1. **Atom Count Limit**: 5-character atom serial column $\implies$ max 99,999 atoms.
2. **Chain Identifier Limit**: 1-character column $\implies$ max 62 chains.
3. Complex structures (e.g. Ribosome 70S, SARS-CoV-2 spike complexes, Vault nanoparticles) exceed these limits and CANNOT be stored in legacy PDB format!

### The Modern Standard: mmCIF (Macromolecular Crystallographic Information File):
- Standardized by wwPDB since 2014.
- Uses STAR (Self-defining Text Archive and Retrieval) grammar.
- Flexible key-value loops (`_atom_site.*`) with NO limits on atom or chain counts.
- Parsed in Biopython with `Bio.PDB.MMCIFParser`.

### 4.3.2 Worked Example

In [ ]:
# 5B.3 Worked Example: Comparing PDB and mmCIF Parsers on 1TUP
cif_file = DATA_DIR / "structures" / "1tup.cif"

cif_parser = MMCIFParser(QUIET=True)
structure_cif = cif_parser.get_structure("1TUP_CIF", cif_file)

pdb_atoms = list(structure.get_atoms())
cif_atoms = list(structure_cif.get_atoms())

print(f"PDB format total atoms:   {len(pdb_atoms):,}")
print(f"mmCIF format total atoms: {len(cif_atoms):,}")

# Verify coordinate concordance on the first Alpha Carbon (CA) in Chain A
ca_pdb = structure[0]["A"][96]["CA"].get_coord()
ca_cif = structure_cif[0]["A"][96]["CA"].get_coord()

print(f"Chain A CA (res 96) PDB coord: {ca_pdb}")
print(f"Chain A CA (res 96) CIF coord: {ca_cif}")
print(f"Euclidean coordinate difference: {np.linalg.norm(ca_pdb - ca_cif):.6f} Å")

### 4.3.3 ✏️ Try It: Atom Count Comparator

Write a function `compare_atom_counts(pdb_p: Path, cif_p: Path) -> dict`:
- Parse `pdb_p` with `PDBParser(QUIET=True)` and `cif_p` with `MMCIFParser(QUIET=True)`.
- Return a dictionary: `{"pdb_atoms": count_pdb, "cif_atoms": count_cif, "match": count_pdb == count_cif}`.

In [ ]:
def compare_atom_counts(pdb_p: Path, cif_p: Path) -> dict:
    p_pdb = PDBParser(QUIET=True).get_structure("pdb", pdb_p)
    p_cif = MMCIFParser(QUIET=True).get_structure("cif", cif_p)
    n_pdb = len(list(p_pdb.get_atoms()))
    n_cif = len(list(p_cif.get_atoms()))
    return {"pdb_atoms": n_pdb, "cif_atoms": n_cif, "match": n_pdb == n_cif}

#### 💡 Explanation & Design Notes

Modern structural bioinformatics tools (AlphaFold, ESMFold, CryoSPARC) operate natively on mmCIF files (`.cif`). Ensuring coordinate agreement across parsers is essential for computational reproducibility.

In [ ]:
# ✅ Test — run me. No output means you passed.
res_cmp = compare_atom_counts(DATA_DIR / "structures" / "1tup.pdb", DATA_DIR / "structures" / "1tup.cif")
assert res_cmp["pdb_atoms"] > 0
assert res_cmp["cif_atoms"] > 0
assert res_cmp["match"]
print("Section 5B.3 passed ✔")

### 4.3.4 ⚠️ Common Mistakes

- **Assuming mmCIF chain IDs are always 1 letter**: In mmCIF, chain IDs can be multi-letter strings (e.g. `AA`, `AB`, `Chain1`).
- **Attempting to load large complexes with PDBParser**: Structures like intact ribosomes will crash or truncate under `PDBParser`. Always use `MMCIFParser` for modern macromolecular assemblies.

## 5B.4 KEGG Biological Pathways & Functional Annotation

### 4.4.1 Theory & Biological Context

The **Kyoto Encyclopedia of Genes and Genomes (KEGG)** integrates genomic, chemical, and systemic biological information.

### Primary KEGG Database Categories:
- **`pathway`**: Higher-order functional wiring diagrams (e.g. `hsa04115`).
- **`genes`**: Gene catalogs across sequenced genomes (e.g. `hsa:7157`).
- **`compound`**: Small chemical molecules and metabolites (e.g. `C00002` for ATP).
- **`disease`**: Human diseases mapped to genetic factors.

### Accessing KEGG via `Bio.KEGG.REST`:
- `KEGG_REST.kegg_find(database, query)`: Search by keyword.
- `KEGG_REST.kegg_get(kegg_id)`: Retrieve full flat-file entry.
- `KEGG_REST.kegg_link(target_db, source_db)`: Cross-link databases (e.g. genes in a pathway).
- `KEGG_REST.kegg_list(database)`: Enumerate all entries.

### 4.4.2 Worked Example

In [ ]:
# 5B.4 Worked Example: Querying KEGG REST API for p53 Signaling Pathway
try:
    print("Searching KEGG for p53 pathways...")
    search_handle = KEGG_REST.kegg_find("pathway", "p53")
    search_text = search_handle.read()
    search_handle.close()
    
    print("Found pathways:")
    for line in search_text.strip().split("\n")[:3]:
        print(" ", line)
        
    # Fetch human p53 pathway entry: hsa04115
    print("Fetching pathway entry: hsa04115 (p53 signaling pathway - Homo sapiens)...")
    get_handle = KEGG_REST.kegg_get("hsa04115")
    pathway_entry = get_handle.read()
    get_handle.close()
    
    # Parse gene lines from pathway entry
    gene_lines = [l for l in pathway_entry.split("\n") if l.startswith("GENE") or (l.startswith(" ") and ";" in l)]
    print(f"Extracted {len(gene_lines)} gene annotations from pathway entry.")
except Exception as e:
    print(f"KEGG network call skipped ({e}). Using offline mock data.")
    pathway_entry = """ENTRY       hsa04115                  Pathway
NAME        p53 signaling pathway - Homo sapiens (human)
GENE        7157  TP53; tumor protein p53
            1026  CDKN1A; cyclin dependent kinase inhibitor 1A
            4193  MDM2; MDM2 proto-oncogene
"""

### 4.4.3 ✏️ Try It: KEGG Pathway Gene Parser

Write a function `parse_kegg_pathway_genes(entry_text: str) -> dict[str, str]`:
- Parses lines under the `GENE` section in a KEGG pathway text file.
- Extracts NCBI Gene ID as the key and Gene Symbol (first word before semicolon) as the value.
- Example: `"7157  TP53; tumor protein p53"` -> `{"7157": "TP53"}`.
- Return the dictionary.

In [ ]:
def parse_kegg_pathway_genes(entry_text: str) -> dict[str, str]:
    genes = {}
    in_gene_section = False
    for line in entry_text.splitlines():
        if line.startswith("GENE"):
            in_gene_section = True
            line = line[4:].strip()
        elif in_gene_section and line.startswith(" ") and ";" in line:
            line = line.strip()
        elif in_gene_section and (not line.startswith(" ") or line.startswith("COMPOUND")):
            in_gene_section = False
            continue
        else:
            continue
            
        # Parse gene line: e.g. "7157  TP53; tumor protein p53"
        parts = line.split(";", 1)
        if len(parts) >= 1:
            tokens = parts[0].split()
            if len(tokens) >= 2:
                gid = tokens[0]
                symbol = tokens[1]
                genes[gid] = symbol
    return genes

#### 💡 Explanation & Design Notes

Parsing KEGG pathway records programmatically extracts functional gene lists for gene set enrichment analysis (GSEA) and over-representation analysis (ORA).

In [ ]:
# ✅ Test — run me. No output means you passed.
kegg_genes = parse_kegg_pathway_genes(pathway_entry)
assert "7157" in kegg_genes
assert kegg_genes["7157"] == "TP53"
print("Section 5B.4 passed ✔")

### 4.4.4 ⚠️ Common Mistakes

- **Hardcoding kegg pathway IDs without organism prefix**: KEGG pathway IDs require organism prefixes (`hsa04115` for human, `mmu04115` for mouse, `map04115` for generic reference map).
- **Violating KEGG rate limits**: KEGG is hosted in Japan with strict robotic request monitoring; always pause (`time.sleep(0.5)`) between consecutive pathway downloads.

## 5B.5 Integrated Multi-Database Pipeline (Gene to Phenotype)

### 4.5.1 Theory & Biological Context

A hallmark of senior bioinformatics engineering is cross-database data integration.
A single biological entity can be comprehensively profiled by traversing primary databases:

```
Gene Symbol: TP53
  ├── NCBI Gene: 7157 (Genomic coordinates: chr17:7,668,402-7,687,550)
  ├── UniProt: P04637 (P53_HUMAN, 393 aa, DNA-binding transcription factor)
  ├── PDB: 1TUP (Crystal structure of core domain complexed with DNA)
  └── KEGG: hsa04115 (p53 signaling pathway)
```

### 4.5.2 Worked Example

In [ ]:
# 5B.5 Worked Example: Building an Integrated Multi-Database Biological Profile
def build_tp53_profile() -> dict:
    profile = {
        "gene_symbol": "TP53",
        "ncbi_gene_id": "7157",
        "uniprot_accession": "P04637",
        "pdb_structure": "1TUP",
        "kegg_pathway": "hsa04115",
        "features": {
            "organism": "Homo sapiens",
            "chromosome": "chr17",
            "protein_length_aa": 393,
            "structure_method": "X-ray Diffraction (2.20 Å)",
            "key_mutational_hotspots": ["R175", "G245", "R248", "R249", "R273", "R282"]
        }
    }
    return profile

tp53_prof = build_tp53_profile()
print("Integrated Biological Profile:")
for k, v in tp53_prof.items():
    print(f"  {k}: {v}")

### 4.5.3 ✏️ Try It: Cross-Reference Validator

Write a function `validate_cross_references(profile: dict) -> bool`:
- Checks that `profile` contains non-empty strings for `'gene_symbol'`, `'ncbi_gene_id'`, `'uniprot_accession'`, `'pdb_structure'`, and `'kegg_pathway'`.
- Return `True` if all 5 exist and are non-empty, else `False`.

In [ ]:
def validate_cross_references(profile: dict) -> bool:
    keys = ["gene_symbol", "ncbi_gene_id", "uniprot_accession", "pdb_structure", "kegg_pathway"]
    return all(bool(profile.get(k)) for k in keys)

#### 💡 Explanation & Design Notes

Cross-referencing validation ensures automated pipeline outputs contain complete linkage metadata before persisting to relational databases or knowledge graphs.

In [ ]:
# ✅ Test — run me. No output means you passed.
assert validate_cross_references(tp53_prof)
assert not validate_cross_references({"gene_symbol": "TP53"})
print("Section 5B.5 passed ✔")

### 4.5.4 ⚠️ Common Mistakes

- **Assuming 1:1 mapping between Gene and PDB structure**: A single gene like TP53 has hundreds of distinct PDB structures (apo, DNA-bound, mutant, peptide-inhibited). Never assume a single canonical structure.
- **Ignoring organism specificity across databases**: Gene names like BRCA1 or TP53 exist across mouse, rat, and zebrafish; always enforce organism filters (`Homo sapiens` / `9606`).

### Practice Exercise 5B.1 (★) — Structural Chain Classification in 1TUP

**Problem**: Write a function `classify_structure_chains(pdb_path: Path) -> dict[str, str]`:
- Parse the structure at `pdb_path` using `Bio.PDB.PDBParser(QUIET=True)`.
- For each chain in model 0:
  - If the chain contains standard amino acids (e.g. parsed by `PPBuilder().build_peptides(chain)` returning non-empty), classify as `"protein"`.
  - Otherwise, classify as `"nucleic_acid"` or `"ligand"`.
- Return a dictionary mapping `{chain_id: classification}`.


- **Input**: Path to 1tup.pdb
- **Expected Output**: Dictionary mapping chains A, B, C to 'protein' and D, E to 'nucleic_acid'

In [ ]:
def classify_structure_chains(pdb_path: Path) -> dict[str, str]:
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("complex", pdb_path)
    model = struct[0]
    ppb = PPBuilder()
    
    classes = {}
    for chain in model:
        peptides = ppb.build_peptides(chain)
        if len(peptides) > 0 and len(peptides[0]) > 10:
            classes[chain.id] = "protein"
        else:
            # Check for nucleic acid residues
            resnames = [r.get_resname().strip() for r in chain.get_residues()]
            if any(r in ["DA", "DC", "DG", "DT", "A", "C", "G", "T", "U"] for r in resnames):
                classes[chain.id] = "nucleic_acid"
            else:
                classes[chain.id] = "other"
    return classes

#### 💡 Solution Walkthrough (5B.1)

1TUP is a multi-molecular co-crystal consisting of a p53 trimer (Chains A, B, C) bound to a double-stranded DNA target consisting of two complementary oligonucleotide strands (Chains E, F).

In [ ]:
# ✅ Test — run me. No output means you passed.
chain_map = classify_structure_chains(DATA_DIR / "structures" / "1tup.pdb")
assert chain_map["A"] == "protein"
assert chain_map["B"] == "protein"
assert chain_map["C"] == "protein"
assert chain_map["E"] == "nucleic_acid"
assert chain_map["F"] == "nucleic_acid"
print("Ex 5B.1 passed ✔")

### Practice Exercise 5B.2 (★) — Batch UniProt Query for DNA Mismatch Repair Complex

**Problem**: Write a function `query_repair_proteins(genes: list[str]) -> pd.DataFrame`:
- Query UniProtKB REST API or simulate for the given list of human gene symbols (`reviewed:true AND gene:... AND organism_id:9606`).
- Return a pandas DataFrame sorted by `'Length'` descending with columns `['Entry', 'Gene', 'Length']`.


- **Input**: List of gene symbols (e.g. ['MLH1', 'MSH2', 'MSH6', 'PMS2'])
- **Expected Output**: DataFrame with columns Entry, Gene, Length sorted by Length descending

In [ ]:
def query_repair_proteins(genes: list[str]) -> pd.DataFrame:
    rows = []
    # Offline dictionary for guaranteed fallback
    known_mmr = {
        "MSH6": {"Entry": "P52701", "Length": 1360},
        "MSH2": {"Entry": "P43246", "Length": 934},
        "PMS2": {"Entry": "P54278", "Length": 862},
        "MLH1": {"Entry": "P40692", "Length": 756}
    }
    
    for g in genes:
        try:
            url = "https://rest.uniprot.org/uniprotkb/search"
            params = {"query": f"reviewed:true AND gene:{g} AND organism_id:9606", "format": "tsv", "fields": "accession,length", "size": 1}
            r = requests.get(url, params=params, timeout=5)
            df = pd.read_csv(io.StringIO(r.text), sep="\t")
            if not df.empty:
                rows.append({"Entry": df.iloc[0]["Entry"], "Gene": g, "Length": int(df.iloc[0]["Length"])})
                continue
        except Exception:
            pass
        # Fallback
        info = known_mmr.get(g, {"Entry": "UNKNOWN", "Length": 500})
        rows.append({"Entry": info["Entry"], "Gene": g, "Length": info["Length"]})
        
    res_df = pd.DataFrame(rows).sort_values(by="Length", ascending=False).reset_index(drop=True)
    return res_df

#### 💡 Solution Walkthrough (5B.2)

DNA mismatch repair proteins form heterodimers (MutSalpha = MSH2-MSH6, MutLalpha = MLH1-PMS2) essential for genome stability. MSH6 is the largest subunit (~1360 aa).

In [ ]:
# ✅ Test — run me. No output means you passed.
df_mmr = query_repair_proteins(["MLH1", "MSH2", "MSH6", "PMS2"])
assert len(df_mmr) == 4
assert list(df_mmr.columns) == ["Entry", "Gene", "Length"]
assert df_mmr["Length"].iloc[0] >= df_mmr["Length"].iloc[-1]
print("Ex 5B.2 passed ✔")

### Practice Exercise 5B.3 (★★) — C-Alpha Residue Distance Matrix & Contact Map

**Problem**: Write a function `compute_chain_contact_map(pdb_path: Path, chain_id: str, threshold_angstrom: float = 8.0) -> np.ndarray`:
- Parse structure at `pdb_path` with `PDBParser(QUIET=True)`.
- Extract all standard residues from `structure[0][chain_id]` that have an `'CA'` (Alpha Carbon) atom.
- Compute the $N \times N$ pairwise Euclidean distance matrix between all $C_\alpha$ atoms.
- A contact is defined as $d \le \text{threshold_angstrom}$.
- Return a binary contact matrix ($1$ for contact, $0$ otherwise) of shape $(N, N)$.


- **Input**: Path to PDB file, chain ID string, and distance threshold float
- **Expected Output**: Binary NumPy array of shape (N, N) where N is number of CA atoms

In [ ]:
def compute_chain_contact_map(pdb_path: Path, chain_id: str, threshold_angstrom: float = 8.0) -> np.ndarray:
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("complex", pdb_path)
    chain = struct[0][chain_id]
    
    ca_atoms = []
    for r in chain.get_residues():
        if r.id[0] == " " and "CA" in r:
            ca_atoms.append(r["CA"])
            
    n = len(ca_atoms)
    coords = np.array([a.get_coord() for a in ca_atoms])
    
    # Compute pairwise Euclidean distance matrix
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    dist_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))
    
    # Threshold into binary contact map
    contact_map = (dist_matrix <= threshold_angstrom).astype(int)
    return contact_map

#### 💡 Solution Walkthrough (5B.3)

Residue contact maps are the fundamental representation used in protein structure prediction (AlphaFold) to evaluate predicted vs experimental 3D folds.

In [ ]:
# ✅ Test — run me. No output means you passed.
cmap = compute_chain_contact_map(DATA_DIR / "structures" / "1tup.pdb", "A", threshold_angstrom=8.0)
assert isinstance(cmap, np.ndarray)
assert cmap.shape[0] == cmap.shape[1]
assert cmap.shape[0] > 180  # p53 core domain has ~190 resolved residues
assert np.all(np.diag(cmap) == 1)  # Every residue is 0 Å from itself
print("Ex 5B.3 passed ✔")

### Practice Exercise 5B.4 (★★) — KEGG Pathway Gene Network Extraction

**Problem**: Write a function `get_kegg_pathway_table(pathway_id: str) -> pd.DataFrame`:
- Fetch pathway entry text using `Bio.KEGG.REST.kegg_get(pathway_id)` (or parse local simulation if network unavailable).
- Extract all genes listed under `GENE`: NCBI Gene ID, Gene Symbol, and Description.
- Return a pandas DataFrame with columns `['gene_id', 'symbol', 'description']`.


- **Input**: KEGG pathway identifier (e.g. 'hsa04115')
- **Expected Output**: DataFrame of pathway genes with columns gene_id, symbol, description

In [ ]:
def get_kegg_pathway_table(pathway_id: str) -> pd.DataFrame:
    try:
        handle = KEGG_REST.kegg_get(pathway_id)
        raw = handle.read()
        handle.close()
    except Exception:
        raw = """GENE        7157  TP53; tumor protein p53
            1026  CDKN1A; cyclin dependent kinase inhibitor 1A
            4193  MDM2; MDM2 proto-oncogene
            1647  GADD45A; growth arrest and DNA damage inducible alpha
            581   BAX; BCL2 associated X, apoptosis regulator
"""
    rows = []
    in_genes = False
    for line in raw.splitlines():
        if line.startswith("GENE"):
            in_genes = True
            line = line[4:].strip()
        elif in_genes and line.startswith(" ") and ";" in line:
            line = line.strip()
        elif in_genes and (not line.startswith(" ") or line.startswith("COMPOUND")):
            in_genes = False
            continue
        else:
            continue
            
        parts = line.split(";", 1)
        if len(parts) == 2:
            left_tokens = parts[0].split()
            desc = parts[1].strip()
            if len(left_tokens) >= 2:
                gid = left_tokens[0]
                sym = left_tokens[1]
                rows.append({"gene_id": gid, "symbol": sym, "description": desc})
                
    return pd.DataFrame(rows)

#### 💡 Solution Walkthrough (5B.4)

This function automates pathway gene member harvesting, a critical step for downstream functional enrichment tests and pathway diagram generation.

In [ ]:
# ✅ Test — run me. No output means you passed.
df_kegg = get_kegg_pathway_table("hsa04115")
assert len(df_kegg) >= 5
assert "7157" in df_kegg["gene_id"].values
assert "TP53" in df_kegg["symbol"].values
print("Ex 5B.4 passed ✔")

### Practice Exercise 5B.5 (★★★) — Structural Biology Interface: DNA-Binding Contact Residues

**Problem**: Write a function `find_dna_binding_residues(pdb_path: Path, protein_chain: str, dna_chains: list[str], distance_cutoff: float = 4.5, out_csv: Path = None) -> pd.DataFrame`:
- Parse structure at `pdb_path` with `PDBParser(QUIET=True)`.
- Find all protein residues in `protein_chain` where at least one atom is within `distance_cutoff` Å of ANY atom in `dna_chains`.
- For each interacting residue, record:
  - `'residue_number'`: residue sequence integer (`r.id[1]`)
  - `'residue_name'`: 3-letter amino acid name (e.g. `'ARG'`)
  - `'min_distance_angstrom'`: minimum distance to any DNA atom
- If `out_csv` is provided, save the results as a CSV file.
- Return the DataFrame sorted by `min_distance_angstrom` ascending.


- **Input**: Path to 1tup.pdb, protein chain ID 'A', DNA chain IDs ['D', 'E'], distance cutoff float, and output CSV Path
- **Expected Output**: DataFrame of interacting residues sorted by min_distance_angstrom

In [ ]:
def find_dna_binding_residues(pdb_path: Path, protein_chain: str, dna_chains: list[str], distance_cutoff: float = 4.5, out_csv: Path = None) -> pd.DataFrame:
    parser = PDBParser(QUIET=True)
    struct = parser.get_structure("complex", pdb_path)
    model = struct[0]
    prot = model[protein_chain]
    
    # Gather all DNA atoms
    dna_atoms = []
    for c_id in dna_chains:
        if c_id in model:
            for r in model[c_id].get_residues():
                for a in r.get_atoms():
                    dna_atoms.append(a)
                    
    dna_coords = np.array([a.get_coord() for a in dna_atoms])
    
    contact_records = []
    for res in prot.get_residues():
        if res.id[0] != " ": continue
        res_atoms = list(res.get_atoms())
        if not res_atoms: continue
        res_coords = np.array([a.get_coord() for a in res_atoms])
        
        # Compute min distance from this residue to any DNA atom
        diff = res_coords[:, np.newaxis, :] - dna_coords[np.newaxis, :, :]
        dists = np.sqrt(np.sum(diff ** 2, axis=-1))
        min_dist = float(np.min(dists))
        
        if min_dist <= distance_cutoff:
            contact_records.append({
                "residue_number": int(res.id[1]),
                "residue_name": res.get_resname().strip(),
                "min_distance_angstrom": round(min_dist, 3)
            })
            
    df_contacts = pd.DataFrame(contact_records).sort_values(by="min_distance_angstrom").reset_index(drop=True)
    if out_csv:
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        df_contacts.to_csv(out_csv, index=False)
    return df_contacts

#### 💡 Solution Walkthrough (5B.5)

This structural bioinformatics algorithm identifies direct macromolecular interface contacts. Cancer hotspot mutations in p53 (such as R248 and R273) directly disrupt these specific electrostatic and hydrogen-bonding interactions with the DNA backbone.

In [ ]:
# ✅ Test — run me. No output means you passed.
out_contacts = OUT_DIR / "p53_dna_contacts.csv"
df_con = find_dna_binding_residues(
    DATA_DIR / "structures" / "1tup.pdb",
    protein_chain="B",
    dna_chains=["E", "F"],
    distance_cutoff=4.5,
    out_csv=out_contacts
)
assert out_contacts.exists()
assert len(df_con) > 0
# Check that known major groove contact residues (Arg248 or Arg273 or Lys120) appear!
contact_res_nums = df_con["residue_number"].values
assert any(r in contact_res_nums for r in [248, 273, 280, 120]), "Known p53 DNA-binding residues must be detected!"
print("Ex 5B.5 passed ✔")

## 6. Summary & Cheat Sheet

UniProt, PDB, and KEGG represent the triumvirate of structural and functional bioinformatics. By combining REST API requests with Biopython's SMCRA structural hierarchy and pathway parsing, complex biological phenomena can be investigated from primary sequence to 3D atomic interfaces.

### Quick Reference Table

| Concept / Operation | Syntax / Function | Biological Application | Key Parameter / Note |
| --- | --- | --- | --- |
| UniProt REST | requests.get('https://rest.uniprot.org/...') | Protein sequence & functional curation | Filter reviewed:true & organism_id:9606 |
| SMCRA Hierarchy | Structure -> Model -> Chain -> Residue -> Atom | 3D structural coordinate model | atom1 - atom2 computes distance in Å |
| Polypeptides | PPBuilder().build_peptides(chain) | Extract AA sequence from 3D coords | Skips missing loop regions |
| PDB Parser | PDBParser(QUIET=True) | Parse legacy .pdb files | Limited to 99,999 atoms |
| mmCIF Parser | MMCIFParser(QUIET=True) | Parse modern .cif files | Unlimited atoms and chains |
| KEGG REST | Bio.KEGG.REST.kegg_get('hsa04115') | Biochemical pathway maps & gene sets | Organism prefix required (hsa, mmu) |

## 7. Further Reading & Resources

- [UniProt REST API Guide](https://www.uniprot.org/help/api): Official API query syntax and field reference.
- [Biopython Tutorial: Chapter 11 (Bio.PDB)](https://biopython.org/DIST/docs/tutorial/Tutorial.html#sec142): The standard manual for macromolecular manipulation in Python.
- [RCSB Protein Data Bank Documentation](https://www.rcsb.org/docs): Structure validation, resolution, and coordinate formats.
- [KEGG: Kyoto Encyclopedia of Genes and Genomes](https://www.kegg.jp/kegg/rest/keggapi.html): KEGG REST API official manual.